## Evaluation (memory-safe retrieval on NCVR)

This notebook evaluates retrieval without persisting new files or loading everything into memory. It uses micro-sharded candidate encoding and top-K tracking across shards.

Metrics: Recall@1/10/50, MRR@10, and PR/F1 via threshold sweep on top-1 cosine similarity.


In [4]:
from eval_utils import EvalParams, evaluate_box

# Choose boxes: e.g., 8 for validation, 9 for test
VAL_BOX = 8
TEST_BOX = 9

MODEL = "nreimers/MiniLM-L6-H384-uncased"    # this was the original model
#MODEL = "outputs/run_minilm_box0_gpu/final"  # or "outputs/run_minilm_box0_gpu/2000"
DEVICE = "cuda"  # or None/"cpu"
# Keep/adjust the other knobs as you prefer
# Optional overrides
DATA_DIR = None  # defaults to data/north_carolina_voters
DEVICE = None    # 'cuda' or 'cpu'; auto-detect if None

# CPU-friendly knobs
BATCH_SIZE = 32
CANDIDATE_CHUNK_SIZE = 20_000
MAX_LENGTH = 64
MAX_QUERIES = 2_000
MAX_CANDIDATES = 100_000
SHARD_MODULUS = 1000
SHARD_REMAINDER = 0
SOURCES_TO_EVAL = None  # e.g., ["dataset_1.csv"] to limit

params_val = EvalParams(
    box_id=VAL_BOX,
    model_name_or_path=MODEL,
    data_dir=DATA_DIR,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    candidate_chunk_size=CANDIDATE_CHUNK_SIZE,
    max_length=MAX_LENGTH,
    max_queries_per_source=MAX_QUERIES,
    max_candidates_per_source=MAX_CANDIDATES,
    shard_modulus=SHARD_MODULUS,
    shard_remainder=SHARD_REMAINDER,
    sources_to_eval=SOURCES_TO_EVAL,
    compute_hard_metrics=True,
)

params_test = EvalParams(
    box_id=TEST_BOX,
    model_name_or_path=MODEL,
    data_dir=DATA_DIR,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    candidate_chunk_size=CANDIDATE_CHUNK_SIZE,
    max_length=MAX_LENGTH,
    max_queries_per_source=MAX_QUERIES,
    max_candidates_per_source=MAX_CANDIDATES,
    shard_modulus=SHARD_MODULUS,
    shard_remainder=SHARD_REMAINDER,
    sources_to_eval=SOURCES_TO_EVAL,
    compute_hard_metrics=True,
)

params_val, params_test


(EvalParams(box_id=8, model_name_or_path='nreimers/MiniLM-L6-H384-uncased', recid_column='recid', entity_columns=('givenname', 'surname', 'postcode', 'suburb'), data_dir=None, device=None, batch_size=32, candidate_chunk_size=20000, ks=(1, 10, 50), verbose=True, log_every_batches=50, max_queries_per_source=2000, max_candidates_per_source=100000, shard_modulus=1000, shard_remainder=0, sources_to_eval=None, seed=42, max_length=64, compute_hard_metrics=True),
 EvalParams(box_id=9, model_name_or_path='nreimers/MiniLM-L6-H384-uncased', recid_column='recid', entity_columns=('givenname', 'surname', 'postcode', 'suburb'), data_dir=None, device=None, batch_size=32, candidate_chunk_size=20000, ks=(1, 10, 50), verbose=True, log_every_batches=50, max_queries_per_source=2000, max_candidates_per_source=100000, shard_modulus=1000, shard_remainder=0, sources_to_eval=None, seed=42, max_length=64, compute_hard_metrics=True))

In [5]:
# Evaluate on validation box
metrics_val = evaluate_box(params_val)
metrics_val


[eval] Loading and serializing box=8 from /home/nicolas/Documents/record_linkage/data/north_carolina_voters...
[eval]  -> Applying shard filter: recid % 1000 == 0
[eval] Source dataset_1.csv: 88 rows serialized
[eval]  -> Applying shard filter: recid % 1000 == 0
[eval] Source dataset_2.csv: 95 rows serialized
[eval]  -> Applying shard filter: recid % 1000 == 0
[eval] Source dataset_3.csv: 95 rows serialized
[eval]  -> Applying shard filter: recid % 1000 == 0
[eval] Source dataset_4.csv: 81 rows serialized
[eval]  -> Applying shard filter: recid % 1000 == 0
[eval] Source dataset_5.csv: 90 rows serialized
[eval] Loaded model 'nreimers/MiniLM-L6-H384-uncased' on device=cpu
[eval] Sources (selected): ['dataset_1.csv', 'dataset_2.csv', 'dataset_3.csv', 'dataset_4.csv', 'dataset_5.csv']
[eval] Row counts per source (selected): {'dataset_1.csv': 88, 'dataset_2.csv': 95, 'dataset_3.csv': 95, 'dataset_4.csv': 81, 'dataset_5.csv': 90}
[eval] Encoding queries from dataset_1.csv: 88 rows...
[eval]

{'recall@1': 0.3900572981274736,
 'recall@10': 0.40605322228129237,
 'recall@50': 0.42395800106326426,
 'mrr@10': 0.9761641732229969,
 'clf_best_f1': 0.9284314793151847,
 'clf_best_threshold': 0.9999996542930603,
 'clf_precision_at_best_f1': 0.9999999999665425,
 'clf_recall_at_best_f1': 0.8701159684782367,
 'recall@1_hard': 0.049628152873766915,
 'recall@10_hard': 0.09047994565538424,
 'recall@50_hard': 0.14151574221749658,
 'mrr@10_hard': 0.6577777777777778,
 'clf_best_f1_hard': 0.09554765750124547,
 'clf_best_threshold_hard': 0.9959033608436585,
 'clf_precision_at_best_f1_hard': 0.05104683468291411,
 'clf_recall_at_best_f1_hard': 0.9999999996016665}

In [ ]:
# Evaluate on test box (optional)
metrics_test = evaluate_box(params_test)
metrics_test
